In [1]:
#imports
import pandas as pd
import os
from bs4 import BeautifulSoup
import re
from tqdm import tqdm
import ast

In [2]:
df = pd.read_csv('plenaire_verslagen_relevant_sections.csv')
df

,Unnamed: 0,title,body,year,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits,text,relevant_text,word_count,relevant_word_count
0,74,Geannoteerde besluitenlijst ministerraad 28 me...,MINISTERRAAD\nKenmerk : 4206864\nBESLUITENLIJS...,2021,yes,[],['kunstmatige intelligentie'],['kunstmatige intelligentie'],0,2,[],NaN,Conclusies van de coördinatiecommissie d.d. 25...,5180.0,109.0
1,76,Agenda ministerraad 4 juni 2021,MINISTERRAAD\nKenmerk : 3753052\nAGENDA\nVerga...,2021,yes,[],"['algoritmen', 'artificiële intelligentie']","['algoritmen', 'artificiële intelligentie']",0,2,[],NaN,Programma Landelijke Vreemdelingen Voorziening...,1219.0,91.0
2,77,Geannoteerde besluitenlijst ministerraad 4 jun...,MINISTERRAAD\nKenmerk : 4208548\nBESLUITENLIJS...,2021,yes,[],"['ai', 'algoritmen', 'artificiële intelligenti...","['ai', 'algoritmen', 'artificiële intelligenti...",0,6,[],NaN,"1 juni 2021,\nnr.22 (Minister van BZ)\nDe conc...",6588.0,488.0
3,134,Geannoteerde besluitenlijst ministerraad 29 ok...,MINISTERRAAD\nKenmerk : 4232087\nBESLUITENLIJS...,2021,yes,[],['bard'],['bard'],0,2,[],NaN,4. EU-implementatie\na. Wijziging van het Alge...,6316.0,205.0
4,148,Geannoteerde besluitenlijst ministerraad 26 no...,MINISTERRAAD\nKenmerk : 4237648\nBESLUITENLIJS...,2021,yes,[],"['ai', 'artificial intelligence']","['ai', 'artificial intelligence']",0,2,[],NaN,Raad Buitenlandse Zaken (Handel) d.d. 29 novem...,5817.0,328.0
5,258,Geannoteerde besluitenlijst ministerraad 16 se...,MINISTERRAAD\nKenmerk : 4286476\nBESLUITENLIJS...,2022,yes,[],"['ai', 'algoritmen', 'algoritmes']","['ai', 'algoritmen', 'algoritmes']",0,3,[],NaN,De minister van BZK zal de brief aan de Tweede...,4857.0,74.0
6,267,Agenda ministerraad 7 oktober 2022,MINISTERRAAD\nKenmerk : 4291255\nAGENDA\nVerga...,2022,yes,[],"['ai', 'algoritmen', 'algoritmes']","['ai', 'algoritmen', 'algoritmes']",0,5,[],NaN,Nader rapport inzake wijziging van de Wet ter ...,658.0,128.0
7,268,Geannoteerde besluitenlijst ministerraad 7 okt...,MINISTERRAAD\nKenmerk : 4291 387\nBESLUITENLIJ...,2022,yes,[],"['ai', 'algoritmen', 'algoritmes', 'kunstmatig...","['drone', 'drones', 'ai', 'algoritmen', 'algor...",0,16,"['drone', 'drones']",NaN,Buitenlands beleid\na. Conclusies van de coörd...,4033.0,795.0
8,288,Geannoteerde besluitenlijst ministerraad 2 dec...,MINISTERRAAD\nKenmerk : 430252 4 en nr. 377745...,2022,yes,[],"['ai', 'artificial intelligence']","['ai', 'artificial intelligence']",0,2,[],NaN,6 december 2022\n3.1 (mogelijk) Uitvoeringsbes...,3979.0,319.0
9,340,Geannoteerde besluitenlijst ministerraad 17 ma...,MINISTERRAAD\nKenmerk : 4321297\nBESLUITENLIJS...,2023,yes,[],"['ai', 'algoritmen', 'algoritmes', 'artificiël...","['ai', 'algoritmen', 'algoritmes', 'artificiël...",0,4,[],NaN,Instellen Adviescommissie Analytics (Minister ...,3139.0,147.0


In [ ]:
plenaire_verslagen = pd.read_csv("plenaire_verslagen_ai_classified.csv")
# change instances in matched_keywords_all to lists
plenaire_verslagen['matched_keywords_all'] = plenaire_verslagen['matched_keywords_all'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)



In [36]:
plenaire_verslagen['company_hits'] = plenaire_verslagen['company_hits'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [4]:
def extract_text(file_path): 
    
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
    
    # Extract paragraphs
    paragraphs = [p.get_text() for p in soup.find_all('p')]
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    text_with_paragraphs = '\n'.join(paragraphs)

    return text_with_paragraphs

In [7]:
path = r"C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\Tweede Kamer\plenaire verslagen html"
text = extract_text(os.path.join(path, plenaire_verslagen['filename'].iloc[0]))



In [ ]:

# plenaire_verslagen['matched_keywords_all'] = plenaire_verslagen['matched_keywords_all'].apply(
#     lambda x: ast.literal_eval(x) if isinstance(x, str) else x
# )

# --- Functions ---
def split_into_sentences(text):
    if not isinstance(text, str):
        return []
    return re.split(r'(?<=[.!?])[\s\n]+', text)

def count_keywords(sentence, keywords):
    if not keywords:
        return 0
    return sum(1 for keyword in keywords if re.search(fr'\b{re.escape(keyword)}\b', sentence, re.I))

def merge_spans(spans):
    """Merge overlapping or adjacent spans into clusters."""
    if not spans:
        return []
    spans.sort()
    merged = [spans[0]]
    for s, e in spans[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e:  # overlap or adjacency
            merged[-1] = (last_s, max(last_e, e))
        else:
            merged.append((s, e))
    return merged

def extract_relevant_sections(row, text_col='text', keywords_col='matched_keywords_all', context_window=2):
    """
    Extract multiple keyword clusters with +/- context_window sentences.
    Returns blocks separated by blank lines.
    """
    text = row[text_col]
    keywords = row[keywords_col]

    if not isinstance(text, str):
        return ""
    if keywords is None or isinstance(keywords, float):
        keywords = []
    if isinstance(keywords, str):
        keywords = [keywords]

    sentences = split_into_sentences(text)
    if not sentences or not keywords:
        return ""

    relevant_indices = [i for i, s in enumerate(sentences) if count_keywords(s, keywords) > 0]
    if not relevant_indices:
        return ""

    # Build spans around each relevant index
    spans = []
    for i in relevant_indices:
        start = max(0, i - context_window)
        end = min(len(sentences), i + context_window + 1)
        spans.append((start, end))

    # Merge overlapping spans into clusters
    merged_spans = merge_spans(spans)

    # Collect blocks
    blocks = [" ".join(sentences[s:e]) for s, e in merged_spans]
    return "\n\n".join(blocks)



# --- Example one-row DataFrame ---
df = pd.DataFrame([{
    "filename": plenaire_verslagen['filename'].iloc[0],
    "text": text,
    "matched_keywords_all": plenaire_verslagen['matched_keywords_all'].iloc[0]
}])

# Apply function
df['relevant_text'] = df.apply(extract_relevant_sections, axis=1)

print(df[['filename', 'relevant_text']])

                                            filename  \
0  kamerstukken-plenaire_verslagen-detail-2014-20...   

                                       relevant_text  
0  Het is nu eind januari. Wanneer komt de minist...  


In [37]:
test = plenaire_verslagen.copy()

test['matched_keywords_all'] = test.apply(
    lambda row: (row['company_hits'] or []) + (row['matched_keywords_all'] or []),
    axis=1
)
test.head()

,filename,text,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits,relevant_text,word_count,relevant_word_count
0,kamerstukken-plenaire_verslagen-detail-2014-20...,,yes,[],['gemini'],[gemini],0,1060,[],Het is nu eind januari. Wanneer komt de minist...,16810683.0,100888.0
1,kamerstukken-plenaire_verslagen-detail-2014-20...,,yes,[],['gemini'],"[asml, gemini]",0,185,[asml],"Die 5,5 miljard haalt zij er ook bij, maar dat...",8707194.0,12765.0
2,kamerstukken-plenaire_verslagen-detail-2015-20...,,yes,[],['algoritmen'],"[google, algoritmen]",0,274,[google],Het is verder aan de Kansspelautoriteit om sam...,18539294.0,23838.0
3,kamerstukken-plenaire_verslagen-detail-2015-20...,,yes,[],['robot'],"[apple, facebook, robot]",0,44,"[apple, facebook]",Het Nederlandse start-up ecosysteem staat hoog...,9425317.0,3168.0
4,kamerstukken-plenaire_verslagen-detail-2015-20...,,yes,[],['robot'],"[drones, google, tesla, uber, robot]",0,39,"[drones, google, tesla, uber]",Mijn eerste punt is dat het curriculum van van...,15082880.0,3939.0


In [38]:
plenaire_verslagen = test.copy()

In [40]:

# Make sure tqdm works with pandas iteration
tqdm.pandas()

for idx, row in tqdm(plenaire_verslagen.iterrows(), total=plenaire_verslagen.shape[0], desc="Processing rows"):
    file_path = os.path.join(path, row['filename'])

    # Extract text from file
    text = extract_text(file_path)

    # Extract relevant sections
    relevant = extract_relevant_sections({
        "text": text,
        "matched_keywords_all": row['matched_keywords_all']
    })

    # Write results back into the DataFrame
    plenaire_verslagen.at[idx, 'text'] = ""  # discard full text to save memory
    plenaire_verslagen.at[idx, 'relevant_text'] = relevant
    plenaire_verslagen.at[idx, 'word_count'] = len(text.split())
    plenaire_verslagen.at[idx, 'relevant_word_count'] = len(relevant.split())

# Save to CSV
plenaire_verslagen.to_csv(
    r"C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\Tweede Kamer\plenaire_verslagen_relevant_sections.csv",
    index=False
)

Processing rows: 100%|██████████| 221/221 [1:57:41<00:00, 31.95s/it]  


In [41]:
plenaire_verslagen.head()

,filename,text,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total,company_hits,relevant_text,word_count,relevant_word_count
0,kamerstukken-plenaire_verslagen-detail-2014-20...,,yes,[],['gemini'],[gemini],0,1060,[],Het is nu eind januari. Wanneer komt de minist...,16810683.0,100888.0
1,kamerstukken-plenaire_verslagen-detail-2014-20...,,yes,[],['gemini'],"[asml, gemini]",0,185,[asml],Ik heb daar met de collega-woordvoerder van me...,8707194.0,26805.0
2,kamerstukken-plenaire_verslagen-detail-2015-20...,,yes,[],['algoritmen'],"[google, algoritmen]",0,274,[google],Eerder is gesproken over het belang van jonger...,18539294.0,33474.0
3,kamerstukken-plenaire_verslagen-detail-2015-20...,,yes,[],['robot'],"[apple, facebook, robot]",0,44,"[apple, facebook]",Wij voeren kritische gesprekken. Het is belang...,9425317.0,67336.0
4,kamerstukken-plenaire_verslagen-detail-2015-20...,,yes,[],['robot'],"[drones, google, tesla, uber, robot]",0,39,"[drones, google, tesla, uber]",Mijn eerste punt is dat het curriculum van van...,15082880.0,94325.0


In [42]:
plenaire_verslagen['relevant_text'].iloc[1]

"Ik heb daar met de collega-woordvoerder van mevrouw Van Veldhoven eens een motie over ingediend: zo'n green bank zou ook voor Nederland een heel mooie manier zijn om dit soort projecten te financieren. In Italië heeft recentelijke een econome, Mazzucato, een prachtig verhaal geschreven over de wijze waarop de entrepreneurial state, de ondernemende staat, kan opereren en geld kan worden terugverdiend. Ze wijst ook op een aantal projecten in de Verenigde Staten, maar je kunt in Nederland bijvoorbeeld ook ASML nemen. ASML is een heel succesvol bedrijf. Het produceert 80% van de chips op de wereldmarkt. Alle bewindspersonen en woordvoerders economische zaken gaan erlangs.\n\nDie 5,5 miljard haalt zij er ook bij, maar dat is weer wat anders. Als je het goed nakijkt, zie je dat die 5,5 miljard de kosten in totaliteit voor de bestaande 1.000 MW betreft. We hebben al wat op zee staan en er komt nog wat bij, Gemini en Loosduinen. In totaal eindigt dat richting 1.000 MW. Wat daar aan kosten aan